In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./data/raw/customers.csv")

customers.info()

다변량 EDA

In [ ]:
order_items = pd.read_csv("./data/raw/order_items.csv")

order_items.info()

In [ ]:
orders = pd.read_csv("./data/raw/orders.csv")

orders.info()

In [ ]:
products = pd.read_csv("./data/raw/products.csv")

products.info()

In [ ]:
#날짜 타입

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["order_date"].info()

In [ ]:
print(customers["customer_id"].isna().sum())
print(order_items["order_item_id"].isna().sum())
print(orders["order_id"].isna().sum())
print(products["product_id"].isna().sum())

In [ ]:
#키 중복 확인

for frame, key in [(customers, "customer_id"), (order_items, "order_item_id"), (orders, "order_id"), (products, "product_id")]:
    print(key, "결측 개수:", frame[key].isna().sum())

In [ ]:
#파생컬럼 total_price(quantity * unit_price) 를 order_items에 추가
order_items["total_price"] = order_items["quantity"] * order_items["unit_price"]

In [ ]:
#도시, 성별, 나이로 등록 고객의 분포(value_counts, describe)를 확인해 보세요

print(customers["city"].value_counts())
print(customers["gender"].value_counts())
print(customers["age"].describe())

In [ ]:
# 상품 카테고리의 개수와 가격 통계
product_count = products["category"].value_counts(dropna=False).to_frame()
print(type(product_count))
product_count

In [ ]:
products["price"].describe()

In [ ]:
# 상품 카테고리별 분포

category_price = products.groupby("category", dropna=False).agg(
    product_count=("product_id", "size"),
    mean_price=("price", "mean"),
    median_price=("price", "median")
)

print(category_price)

완료 주문 병합 후 검증

In [ ]:
#불리언마스크
#orders["order_status"] == "completed"
#completed_orders = orders.value_counts("order_status",)
completed_orders = orders[orders["order_status"] == "completed"]
print(completed_orders["order_status"].value_counts())

In [ ]:
# order_items와 orders 병합
item_with_orders = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

item_with_orders.head()

In [ ]:
# completed 주문만 추출
completed_item_with_orders = item_with_orders[
    item_with_orders["order_status"] == "completed"
]

completed_item_with_orders.head()

In [ ]:
#고객별 완료 주문 건수
customer_completed_orders = (
    completed_item_with_orders
    .groupby("customer_id")["order_id"]
    .nunique()
)

customer_completed_orders.head()

In [ ]:
#고객별 평균 주문건수(완료건 기준)
customer_completed_orders.mean()

In [ ]:
# 카테고리별 수량 합산하기, 완료주문 금액 합산해보세요
# 카테고리(products), 주문상태 완료(), 금액(order_items)
# 1 completed_items (oders, order_items)
# 2 category_tiems (completed_tiems, products)